# 18_generate_clinician_ready_pdf

Create a clinician friendly qualitative PDF.

Layout per page:
- one selected image
- one row per model/scenario
- columns: original image with lesion outline, XAI 0 Grad-CAM, XAI 1 difference map, XAI 2 Finer-CAM

The CAM panels are generated by `scripts.generate_finer_cam_panderm` using `--clinician_labels`.

In [1]:
from pathlib import Path
import json
import subprocess
import shlex
import pandas as pd

# =============================================================================
# 1. MAIN PARAMETERS TO MODIFY
# =============================================================================

QUAL_SEED = 42
TARGET_BLOCK_INDEX = -4
# NUM_SAMPLES = 1232 # 10
DRY_RUN = False

REPO_ROOT = Path("..").resolve()
HAM_ROOT = REPO_ROOT / "data" / "HAM10000"
IMG_DIR = HAM_ROOT
MASK_ROOT = HAM_ROOT

# Choose one mode:
# - "ham7": main 7-class HAM10000 models for clinician material
# - "mel_nv": binary MEL vs NV cue/control material
MODE = "mel_nv"

# -----------------------------------------------------------------------------
# Mode A: 7-class main models
# -----------------------------------------------------------------------------
# HAM7_CSV = HAM_ROOT / f"ham_test_cam_qualitative_stratified_10_seed{QUAL_SEED}.csv"
HAM7_CSV = HAM_ROOT / "ham_test_cam_all_with_masks_grouped_by_class.csv"
HAM7_GT_COL = "gt_label"
HAM7_CLASS_ARGS = ["--class_preset", "ham"]

# Recommended for clinician examples:
# GT class as target, strongest predicted non-target class as reference.
HAM7_COMPARE_ARGS = [
    "--compare_mode", "gt_topk_non_target",
    "--topk_compare", "1",
]

HAM7_SCENARIOS = [
    # {
    #     "name": "Model 1: Baseline",
    #     "checkpoint": REPO_ROOT / "external" / "checkpoints3" / "checkpoint-best-base.pth",
    #     "checkpoint_model_type": "panderm",
    #     "csv": HAM7_CSV,
    # },
    {
        "name": "Model 2: Human aligned",
        "checkpoint": REPO_ROOT / "external" / "checkpoints3" / "checkpoint-best-HA075.pth",
        "checkpoint_model_type": "panderm",
        "csv": HAM7_CSV,
    },
]

# -----------------------------------------------------------------------------
# Mode B: binary MEL vs NV cue/control models
# -----------------------------------------------------------------------------
CUE_SIZE = "brown"          # small, big, brown
CUE_PLACEMENT = "fixed"     # fixed, random

if CUE_PLACEMENT == "fixed":
    MEL_NV_ROOT = HAM_ROOT / f"synthetic_cue_{CUE_SIZE}" / f"mel_nv_fixed_center_seed{QUAL_SEED}"
    MEL_NV_CUE_CSV = MEL_NV_ROOT / "csv" / f"ham_mel_nv_cue_fixed_qualitative_10_seed{QUAL_SEED}.csv"
elif CUE_PLACEMENT == "random":
    MEL_NV_ROOT = HAM_ROOT / f"synthetic_cue_{CUE_SIZE}" / f"mel_nv_random_location_seed{QUAL_SEED}"
    MEL_NV_CUE_CSV = MEL_NV_ROOT / "csv" / f"ham_mel_nv_cue_random_qualitative_10_seed{QUAL_SEED}.csv"
else:
    raise ValueError("CUE_PLACEMENT must be 'fixed' or 'random'.")

MEL_NV_CLEAN_CSV = HAM_ROOT / f"synthetic_cue_{CUE_SIZE}" / f"mel_nv_fixed_center_seed{QUAL_SEED}" / "csv" / f"ham_mel_nv_clean_qualitative_10_seed{QUAL_SEED}.csv"
MEL_NV_GT_COL = "gt_label"
MEL_NV_CLASS_ARGS = ["--class_names", "MEL,NV"]
MEL_NV_COMPARE_ARGS = [
    "--compare_mode", "gt_pair",
    "--A", "MEL",
    "--B", "NV",
    "--topk_compare", "1",
]

MEL_NV_CHECKPOINT_ROOT = REPO_ROOT / "external" / f"checkpoints2_{CUE_SIZE}"

MEL_NV_SCENARIOS = [
    {
        "name": "Model 1: Clean model on clean image",
        "checkpoint": MEL_NV_CHECKPOINT_ROOT / "checkpoint-best-clean.pth",
        "checkpoint_model_type": "panderm",
        "csv": MEL_NV_CLEAN_CSV,
    },
    {
        "name": "Model 2: Clean model on cued image",
        "checkpoint": MEL_NV_CHECKPOINT_ROOT / "checkpoint-best-clean.pth",
        "checkpoint_model_type": "panderm",
        "csv": MEL_NV_CUE_CSV,
    },
    {
        "name": "Model 3: Cue-trained model",
        "checkpoint": MEL_NV_CHECKPOINT_ROOT / "checkpoint-best-cue.pth",
        "checkpoint_model_type": "panderm",
        "csv": MEL_NV_CUE_CSV,
    },
    {
        "name": "Model 4: Cue + human aligned",
        "checkpoint": MEL_NV_CHECKPOINT_ROOT / "checkpoint-best-cue-ha.pth",
        "checkpoint_model_type": "panderm",
        "csv": MEL_NV_CUE_CSV,
    },
]

# -----------------------------------------------------------------------------
# Active configuration
# -----------------------------------------------------------------------------
if MODE == "ham7":
    ACTIVE_SCENARIOS = HAM7_SCENARIOS
    ACTIVE_GT_COL = HAM7_GT_COL
    ACTIVE_CLASS_ARGS = HAM7_CLASS_ARGS
    ACTIVE_COMPARE_ARGS = HAM7_COMPARE_ARGS
    OUT_ROOT = REPO_ROOT / "outputs" / f"clinician_ready_ham7_seed{QUAL_SEED}_block{TARGET_BLOCK_INDEX}"
    NUM_SAMPLES = len(pd.read_csv(HAM7_CSV))
elif MODE == "mel_nv":
    ACTIVE_SCENARIOS = MEL_NV_SCENARIOS
    ACTIVE_GT_COL = MEL_NV_GT_COL
    ACTIVE_CLASS_ARGS = MEL_NV_CLASS_ARGS
    ACTIVE_COMPARE_ARGS = MEL_NV_COMPARE_ARGS
    OUT_ROOT = REPO_ROOT / "outputs" / f"clinician_ready_mel_nv_{CUE_SIZE}_{CUE_PLACEMENT}_seed{QUAL_SEED}_block{TARGET_BLOCK_INDEX}"
    NUM_SAMPLES = len(pd.read_csv(MEL_NV_CUE_CSV))
else:
    raise ValueError("MODE must be 'ham7' or 'mel_nv'.")

OUT_ROOT.mkdir(parents=True, exist_ok=True)
PANEL_ROOT = OUT_ROOT / "panels"
PDF_OUT = OUT_ROOT / f"clinician_ready_{MODE}_seed{QUAL_SEED}_block{TARGET_BLOCK_INDEX}.pdf"
CONFIG_OUT = OUT_ROOT / f"clinician_ready_config_{MODE}_seed{QUAL_SEED}_block{TARGET_BLOCK_INDEX}.json"

print("MODE:", MODE)
print("TARGET_BLOCK_INDEX:", TARGET_BLOCK_INDEX)
print("OUT_ROOT:", OUT_ROOT)
print("PDF_OUT:", PDF_OUT)
print("Number of scenarios:", len(ACTIVE_SCENARIOS))
print("Number of samples per scenario:", NUM_SAMPLES)

for s in ACTIVE_SCENARIOS:
    print("\n", s["name"])
    print("  checkpoint:", s["checkpoint"])
    print("  csv:", s["csv"])
    if not Path(s["checkpoint"]).exists():
        print("  [WARN] missing checkpoint")
    if not Path(s["csv"]).exists():
        print("  [WARN] missing CSV")

MODE: mel_nv
TARGET_BLOCK_INDEX: -4
OUT_ROOT: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/clinician_ready_mel_nv_brown_fixed_seed42_block-4
PDF_OUT: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/clinician_ready_mel_nv_brown_fixed_seed42_block-4/clinician_ready_mel_nv_seed42_block-4.pdf
Number of scenarios: 4
Number of samples per scenario: 10

 Model 1: Clean model on clean image
  checkpoint: /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints2_brown/checkpoint-best-clean.pth
  csv: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue_brown/mel_nv_fixed_center_seed42/csv/ham_mel_nv_clean_qualitative_10_seed42.csv

 Model 2: Clean model on cued image
  checkpoint: /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints2_brown/checkpoint-best-clean.pth
  csv: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue_brown/mel_nv_fixed_center_seed42/csv/ham_mel_nv_cu

## 2. Helper functions

This cell generates one clinician labeled CAM panel per scenario.

In [2]:
def run_command(cmd: list[str], dry_run: bool = False):
    print("\n" + "=" * 100)
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 100)
    if dry_run:
        return
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def safe_name(name: str) -> str:
    out = str(name).lower()
    for ch in [" ", "/", "\\", ":", ";", ",", "(", ")", "[", "]", "{", "}", "+"]:
        out = out.replace(ch, "_")
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_")


def generate_clinician_panels(
    scenarios: list[dict],
    num_samples: int,
    dry_run: bool = False,
):
    panel_items = "rgb_gt_mask,gradcam_a,map_diff,finercam"

    for scenario in scenarios:
        scenario_name = scenario["name"]
        scenario_out_dir = PANEL_ROOT / safe_name(scenario_name)
        scenario_out_dir.mkdir(parents=True, exist_ok=True)

        cmd = [
            "python", "-m", "scripts.generate_finer_cam_panderm",
            "--csv", str(scenario["csv"]),
            "--image_col", "image_rel_path",
            "--img_dir", str(IMG_DIR),
            "--gt_col", ACTIVE_GT_COL,
            "--checkpoint", str(scenario["checkpoint"]),
            "--checkpoint_model_type", scenario.get("checkpoint_model_type", "panderm"),
            "--out_dir", str(scenario_out_dir),
            "--num_samples", str(num_samples),
            "--method", "finercam",
            "--alpha", "0.8",
            "--panel_items", panel_items,
            "--mask_root", str(MASK_ROOT),
            "--mask_col", "mask_rel_path",
            "--target_block_index", str(TARGET_BLOCK_INDEX),
            "--clinician_labels",
            "--model_display_name", scenario_name,
            # "--save_json",
        ]

        cmd += ACTIVE_CLASS_ARGS
        cmd += ACTIVE_COMPARE_ARGS

        if scenario.get("use_seg_gate", False):
            cmd += [
                "--use_seg_gate",
                "--seg_gate_bg_keep", str(scenario.get("seg_gate_bg_keep", 0.05)),
            ]

        print(f"\nGenerating clinician panels: {scenario_name}")
        run_command(cmd, dry_run=dry_run)


generate_clinician_panels(
    scenarios=ACTIVE_SCENARIOS,
    num_samples=NUM_SAMPLES,
    dry_run=DRY_RUN,
)


Generating clinician panels: Model 1: Clean model on clean image

python -m scripts.generate_finer_cam_panderm --csv /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue_brown/mel_nv_fixed_center_seed42/csv/ham_mel_nv_clean_qualitative_10_seed42.csv --image_col image_rel_path --img_dir /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000 --gt_col gt_label --checkpoint /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints2_brown/checkpoint-best-clean.pth --checkpoint_model_type panderm --out_dir /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/clinician_ready_mel_nv_brown_fixed_seed42_block-4/panels/model_1_clean_model_on_clean_image --num_samples 10 --method finercam --alpha 0.8 --panel_items rgb_gt_mask,gradcam_a,map_diff,finercam --mask_root /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000 --mask_col mask_rel_path --target_block_index -4 --clinician_labels --model_display_name 'Model 1: Clean

## 3. Build clinician ready PDF

This creates one PDF page per selected image. Each page has one row per model/scenario.

In [3]:
from PIL import Image, ImageDraw, ImageFont
import math
import re

PANEL_SUFFIX = "rgb_gt_mask_gradcam_a_map_diff_finercam"


def get_font(size: int, bold: bool = False):
    candidates = [
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for path in candidates:
        if path and Path(path).exists():
            return ImageFont.truetype(path, size=size)
    return ImageFont.load_default()


FONT_TITLE = get_font(34, bold=True)
FONT_SUBTITLE = get_font(22, bold=False)
FONT_LABEL = get_font(24, bold=True)
FONT_SMALL = get_font(18, bold=False)


def image_id_to_stem(image_id_value: str) -> str:
    p = Path(str(image_id_value))
    stem = p.stem if p.suffix else p.name
    return stem.replace("/", "_").replace("\\", "_").replace(" ", "_")


def load_display_rows_for_pdf(scenarios: list[dict]) -> pd.DataFrame:
    # Use the first scenario CSV as page order.
    first_csv = Path(scenarios[0]["csv"])
    df = pd.read_csv(first_csv).head(NUM_SAMPLES).copy()

    if "image_id" not in df.columns:
        if "image_rel_path" in df.columns:
            df["image_id"] = df["image_rel_path"].apply(lambda x: Path(str(x)).stem)
        elif "image" in df.columns:
            df["image_id"] = df["image"].apply(lambda x: Path(str(x)).stem)
        else:
            raise ValueError("Need one of image_id, image_rel_path, or image columns.")

    return df


def find_panel_png(out_dir: Path, row: pd.Series) -> Path | None:
    candidates = []

    if "image_rel_path" in row and pd.notna(row["image_rel_path"]):
        candidates.append(image_id_to_stem(row["image_rel_path"]))
    if "image_id" in row and pd.notna(row["image_id"]):
        candidates.append(image_id_to_stem(row["image_id"]))
    if "image" in row and pd.notna(row["image"]):
        candidates.append(image_id_to_stem(row["image"]))

    for stem in dict.fromkeys(candidates):
        direct = out_dir / f"{stem}_{PANEL_SUFFIX}.png"
        if direct.exists():
            return direct
        matches = sorted(out_dir.glob(f"{stem}_*.png"))
        if matches:
            return matches[0]

    return None


def wrap_text(draw, text: str, font, max_width: int) -> list[str]:
    words = str(text).split()
    lines = []
    current = ""

    for word in words:
        test = f"{current} {word}".strip()
        bbox = draw.textbbox((0, 0), test, font=font)
        if bbox[2] - bbox[0] <= max_width:
            current = test
        else:
            if current:
                lines.append(current)
            current = word

    if current:
        lines.append(current)

    return lines


def make_page_for_image(row: pd.Series, scenarios: list[dict]) -> Image.Image:
    page_width = 2200
    margin = 50
    label_width = 290
    gap = 18
    title_h = 120

    available_panel_width = page_width - 2 * margin - label_width - gap

    loaded_panels = []
    for idx, scenario in enumerate(scenarios, start=1):
        out_dir = PANEL_ROOT / safe_name(scenario["name"])
        panel_path = find_panel_png(out_dir, row)
        if panel_path is None:
            loaded_panels.append((scenario, None, None))
            continue

        panel = Image.open(panel_path).convert("RGB")
        scale = available_panel_width / panel.width
        new_h = int(panel.height * scale)
        panel = panel.resize((available_panel_width, new_h), Image.Resampling.LANCZOS)
        loaded_panels.append((scenario, panel, panel_path))

    row_heights = []
    for _, panel, _ in loaded_panels:
        row_heights.append(panel.height if panel is not None else 260)

    page_height = title_h + margin + sum(row_heights) + gap * (len(row_heights) - 1) + margin
    page = Image.new("RGB", (page_width, page_height), "white")
    draw = ImageDraw.Draw(page)

    image_id = row.get("image_id", row.get("image_rel_path", "unknown"))
    gt = row.get(ACTIVE_GT_COL, row.get("gt_label", "unknown"))
    title = f"Clinician review example: {image_id}"
    subtitle = f"Ground truth: {gt} | Target block: {TARGET_BLOCK_INDEX}"

    draw.text((margin, 30), title, fill="black", font=FONT_TITLE)
    draw.text((margin, 78), subtitle, fill=(60, 60, 60), font=FONT_SUBTITLE)

    y = title_h
    for idx, (scenario, panel, panel_path) in enumerate(loaded_panels, start=1):
        row_h = row_heights[idx - 1]

        label = scenario["name"]
        label_lines = wrap_text(draw, label, FONT_LABEL, label_width - 10)
        label_x = margin
        label_y = y + 25

        for line in label_lines:
            draw.text((label_x, label_y), line, fill="black", font=FONT_LABEL)
            label_y += 32

        draw.text((label_x, label_y + 10), f"Block {TARGET_BLOCK_INDEX}", fill=(80, 80, 80), font=FONT_SMALL)

        if panel is None:
            box_x = margin + label_width + gap
            box_y = y
            draw.rectangle([box_x, box_y, box_x + available_panel_width, box_y + row_h], outline=(180, 180, 180), width=2)
            draw.text((box_x + 30, box_y + 80), "Missing panel PNG", fill=(160, 0, 0), font=FONT_LABEL)
        else:
            page.paste(panel, (margin + label_width + gap, y))

        y += row_h + gap

    return page


def build_clinician_pdf():
    rows = load_display_rows_for_pdf(ACTIVE_SCENARIOS)

    pages = []
    for _, row in rows.iterrows():
        page = make_page_for_image(row, ACTIVE_SCENARIOS)
        pages.append(page)

    if not pages:
        raise RuntimeError("No pages generated.")

    PDF_OUT.parent.mkdir(parents=True, exist_ok=True)
    pages[0].save(PDF_OUT, save_all=True, append_images=pages[1:], resolution=150.0)

    config = {
        "mode": MODE,
        "qual_seed": QUAL_SEED,
        "target_block_index": TARGET_BLOCK_INDEX,
        "num_samples": NUM_SAMPLES,
        "pdf_out": str(PDF_OUT),
        "out_root": str(OUT_ROOT),
        "scenarios": [
            {
                "name": s["name"],
                "checkpoint": str(s["checkpoint"]),
                "csv": str(s["csv"]),
                "checkpoint_model_type": s.get("checkpoint_model_type", "panderm"),
            }
            for s in ACTIVE_SCENARIOS
        ],
        "class_args": ACTIVE_CLASS_ARGS,
        "compare_args": ACTIVE_COMPARE_ARGS,
    }
    CONFIG_OUT.write_text(json.dumps(config, indent=2))

    print("Saved PDF:", PDF_OUT)
    print("Saved config:", CONFIG_OUT)


build_clinician_pdf()

Saved PDF: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/clinician_ready_mel_nv_brown_fixed_seed42_block-4/clinician_ready_mel_nv_seed42_block-4.pdf
Saved config: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/clinician_ready_mel_nv_brown_fixed_seed42_block-4/clinician_ready_config_mel_nv_seed42_block-4.json


## 4. Quick checks

Use this cell to inspect whether panels were created for every scenario.

In [4]:
# for scenario in ACTIVE_SCENARIOS:
#     out_dir = PANEL_ROOT / safe_name(scenario["name"])
#     pngs = sorted(out_dir.glob("*.png"))
#     metas = sorted(out_dir.glob("*_meta.json"))
#     print("\n" + scenario["name"])
#     print("  out_dir:", out_dir)
#     print("  png panels:", len(pngs))
#     print("  meta files:", len(metas))
#     if pngs[:3]:
#         for p in pngs[:3]:
#             print("   ", p.name)